<a href="https://colab.research.google.com/github/Akshatha7710/smart-tea-estate-management-system/blob/soil_quality-analysis_and_predictive_modeling/Soil_Quality_Analysis_and_Predictive_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

class TeaEstateAnalyzer:
    def __init__(self, soil_df, weather_df):
        self.soil_df = soil_df
        self.weather_df = weather_df
        # Optimal tea soil pH is generally between 4.5 and 5.5
        self.PH_LOWER_LIMIT = 4.5
        self.PH_UPPER_LIMIT = 5.5

    def calculate_evapotranspiration(self, temp, humidity=70):
        """
        Refined Feature Engineering: Simple approximation of
        Potential Evapotranspiration (PET).
        """
        # Evaporation scales non-linearly with temperature
        return (temp * 0.12) * (1 - (humidity / 100))

    def predict_moisture_trend(self, field_no):
        field = self.soil_df[self.soil_df['Field_No'] == field_no].iloc[0]
        current_moisture = field['Moisture_Base']

        projections = []
        for _, day in self.weather_df.iterrows():
            evap = self.calculate_evapotranspiration(day['temp'])

            # Effective rainfall: Soils have a 'Field Capacity'
            # If rain is too heavy (>35mm), the rest is surface runoff
            effective_rain = min(day['rain'], 35) + (max(0, day['rain'] - 35) * 0.1)

            # Soil moisture physics: absorption coefficient (approx 0.3 for clay-loam)
            new_moisture = (current_moisture - evap) + (effective_rain * 0.35)

            # Cap moisture at 100% (Saturation) and floor at 5% (Wilting point)
            current_moisture = np.clip(new_moisture, 5, 100)

            projections.append({
                'Date': day['date'],
                'Moisture': round(current_moisture, 2),
                'Rain': day['rain']
            })
        return pd.DataFrame(projections)

    def generate_management_report(self, field_no):
        field = self.soil_df[self.soil_df['Field_No'] == field_no].iloc[0]
        projections = self.predict_moisture_trend(field_no)
        avg_moisture = projections['Moisture'].mean()
        total_rain = projections['Rain'].sum()

        print(f"\n--- 📋 REPORT FOR FIELD {field_no} ---")
        print(f"Status: pH {field['pH']} | Carbon {field['Carbon_Pct']}%")

        # Logic-driven Decision Engine
        if field['pH'] < self.PH_LOWER_LIMIT:
            action = "🚨 STOP: Soil too acidic. Apply Dolomite (Lime) to raise pH."
        elif total_rain > 60:
            action = "🌊 DELAY: Monsoon-level rain forecast. Fertilizer will wash away (Leaching)."
        elif avg_moisture < 18:
            action = "🏜️ IRRIGATION REQUIRED: Soil moisture below 18% prevents nutrient transport."
        elif field['Carbon_Pct'] < 1.8:
            action = "🌱 SUSTAINABLE: Apply Fertilizer + Bio-char/Compost to boost Organic Carbon."
        else:
            action = "✅ OPTIMAL: Proceed with planned NPK application."

        print(f"DECISION: {action}\n")
        print(projections.to_string(index=False))

# --- DATA INITIALIZATION ---
soil_data = {
    'Field_No': [1, 2, 3, 4],
    'pH': [4.4, 4.6, 5.1, 3.8],
    'Carbon_Pct': [1.8, 1.5, 2.1, 1.1],
    'Moisture_Base': [28, 25, 30, 22]
}

forecast = {
    'date': [(datetime.now() + timedelta(days=i)).date() for i in range(5)],
    'rain': [12, 0, 45, 5, 2],
    'temp': [24, 25, 22, 26, 28]
}

# --- EXECUTION ---
analyzer = TeaEstateAnalyzer(pd.DataFrame(soil_data), pd.DataFrame(forecast))
analyzer.generate_management_report(field_no=4)


--- 📋 REPORT FOR FIELD 4 ---
Status: pH 3.8 | Carbon 1.1%
DECISION: 🚨 STOP: Soil too acidic. Apply Dolomite (Lime) to raise pH.

      Date  Moisture  Rain
2025-12-28     25.34    12
2025-12-29     24.44     0
2025-12-30     36.24    45
2025-12-31     37.06     5
2026-01-01     36.75     2
